# SelvaSonic — Análisis de Errores del Baseline (Semana 3)

## Objetivo de este notebook

El cronograma pide responder dos preguntas sobre el modelo baseline:

> **¿Qué confunde el modelo? ¿Por qué?**

Entrenar un modelo y reportar solo el *accuracy* global (`test_acc = 0.632`) es como decir "el paciente está enfermo" sin decir de qué. Un número agregado esconde la historia real: **¿el modelo falla parejo en todas las especies, o hay unas que le cuestan muchísimo y otras que domina?** Este notebook abre esa caja negra.

## Por qué el análisis de errores es la parte más valiosa del proyecto

En Machine Learning aplicado, el análisis de errores es lo que distingue un proyecto de clase de uno de investigación. Las razones:

1. **Guía las mejoras.** No tiene sentido agregar Multi-Head Attention (Semana 4) a ciegas. Si sabemos *qué* confunde el baseline, podemos verificar si el attention ataca justo ese problema.
2. **Distingue errores "honestos" de errores "patológicos".** Confundir dos especies del mismo género (*Crypturellus cinereus* vs *Crypturellus undulatus*) es un error razonable —hasta un ornitólogo podría dudar. Confundir un tucán con ruido de lluvia sería patológico.
3. **Conecta el modelo con el dominio.** Somos ingenieras físicas trabajando en bioacústica: el análisis debe hablar tanto el lenguaje de las métricas (precision, recall, F1) como el del fenómeno (vocalizaciones, taxonomía, solapamiento espectral).

## Estructura

| Sección | Qué responde |
|---|---|
| 1. Setup y carga del modelo | Reconstruir el estado exacto del entrenamiento |
| 2. Predicciones sobre test | Obtener `y_true`, `y_pred`, y las probabilidades |
| 3. Matriz de confusión normalizada | Ver el patrón de confusiones sin sesgo de tamaño |
| 4. F1 vs cantidad de datos | Demostrar que el desbalance explica gran parte del error |
| 5. Errores de alta confianza | Los fallos más "vergonzosos" del modelo |
| 6. Confusiones bioacústicas | Interpretación taxonómica: ¿tiene sentido biológico? |
| 7. Calibración de confianza | ¿El modelo sabe cuándo no sabe? (clave para el umbral de "No identificado") |
| 8. Conclusiones para Semana 4 | Qué debe mejorar el attention |

---
## Sección 1 — Setup y carga del modelo entrenado

### Qué hacemos y por qué

Para analizar errores necesitamos **reproducir exactamente** el conjunto de test que el modelo nunca vio durante el entrenamiento. Esto es crítico: si reconstruimos el split con una semilla distinta, estaríamos evaluando sobre datos que el modelo *sí* pudo haber visto en train, inflando las métricas (data leakage).

La clave está en usar el **mismo `random_state=42`** que usó `create_dataloaders` durante el entrenamiento. Como el split en `dataset.py` es determinista dada la semilla (usa `np.random.default_rng(random_state)`), obtendremos clip por clip el mismo test set.

Cargamos el modelo desde `best.pth` (la época 19, la de mejor `val_acc`), **no** desde `latest.pth` (época 27). Esto es deliberado: el `best.pth` es el modelo que mejor generaliza, antes de que el sobreajuste avanzara más.

In [ ]:
import sys
import os
from pathlib import Path

# Asegurar que src/ es importable (ajusta si ejecutas desde otra carpeta)
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / 'src').exists():
    sys.path.insert(0, str(PROJECT_ROOT))
elif (PROJECT_ROOT.parent / 'src').exists():
    # Si el notebook está dentro de notebooks/, subir un nivel
    PROJECT_ROOT = PROJECT_ROOT.parent
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from src.dataset import create_dataloaders
from src.model import SelvaSonicCNN
from src import config

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {device}')

# Paleta de colores (preferencias de Laura)
COLOR_PRIMARY = '#6C5CE7'   # púrpura
COLOR_ACCENT = '#00CEC9'    # turquesa
COLOR_WARN = '#FD79A8'      # rosa
COLOR_DARK = '#2D3436'

In [ ]:
# Rutas del run baseline
RUN_DIR = PROJECT_ROOT / 'results' / 'runs' / 'baseline_S3_v2_20260527_0118'
BEST_CKPT = RUN_DIR / 'best.pth'
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'

assert BEST_CKPT.exists(), f'No se encontró {BEST_CKPT}. Verifica que descargaste el run de Drive.'
assert RAW_DATA_DIR.exists(), f'No se encontró {RAW_DATA_DIR}. El análisis necesita los audios crudos.'

print(f'Checkpoint: {BEST_CKPT}')
print(f'Datos crudos: {RAW_DATA_DIR}')

### Reconstruir el test set con la MISMA semilla

Usamos `random_state=42` y los mismos ratios `0.70 / 0.15 / 0.15` del entrenamiento. El `verbose=True` nos deja confirmar que el número de clips por split coincide con lo que vimos en el notebook de entrenamiento (`test=1471`).

In [ ]:
# IMPORTANTE: mismos parámetros que en el entrenamiento para reproducir el split exacto
train_loader, val_loader, test_loader, label_map = create_dataloaders(
    raw_data_dir=str(RAW_DATA_DIR),
    batch_size=32,
    num_workers=0,        # 0 en Windows para análisis (evita overhead de multiprocessing)
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    random_state=42,      # <-- LA MISMA SEMILLA DEL ENTRENAMIENTO
    verbose=True,
)

NUM_CLASSES = len(label_map)
# idx -> nombre, ordenado por índice
idx_to_name = {v: k for k, v in label_map.items()}
class_names = [idx_to_name[i] for i in range(NUM_CLASSES)]

print(f'\nNúmero de clases: {NUM_CLASSES}')
print(f'Clases: {class_names}')
print(f'\nEsperado del entrenamiento: test=1471 clips. Verifica arriba que coincida.')

In [ ]:
# Cargar el modelo desde best.pth
ckpt = torch.load(BEST_CKPT, map_location=device)

model = SelvaSonicCNN(num_classes=NUM_CLASSES).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print(f"Modelo cargado desde best.pth")
print(f"  Época del checkpoint : {ckpt['epoch'] + 1}")
print(f"  best_val_acc         : {ckpt['best_val_acc']:.4f}")
print(f"  Parámetros           : {model.count_parameters():,}")
if 'hparams' in ckpt:
    print(f"  Hiperparámetros      : {ckpt['hparams']}")

---
## Sección 2 — Predicciones sobre el conjunto de test

### Qué calculamos

Recorremos el test set una vez y guardamos tres cosas por cada clip:

- `y_true`: la etiqueta real (lo que el clip *es*).
- `y_pred`: la predicción del modelo (`argmax` de los logits).
- `y_probs`: el vector completo de probabilidades (softmax de los logits). Lo necesitamos para el análisis de **confianza** (secciones 5 y 7).

### Por qué guardamos las probabilidades y no solo la predicción

El `argmax` colapsa toda la información: nos dice *qué* eligió el modelo, pero no *con cuánta seguridad*. Un modelo que predice la clase correcta con probabilidad 0.95 es muy distinto de uno que la predice con 0.34 (apenas ganándole a las demás). Esa diferencia es central para:

- Decidir el **umbral de "No identificado"** del proyecto (Semana 5-6): si el modelo no supera cierta confianza, preferimos que diga "no sé" a que se equivoque.
- Entender si los errores son "dudas" (baja confianza) o "convicciones equivocadas" (alta confianza), que son problemas muy distintos.

In [ ]:
import torch.nn.functional as F

all_true = []
all_pred = []
all_probs = []

model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model(x)
        probs = F.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        all_true.append(y.cpu())
        all_pred.append(preds.cpu())
        all_probs.append(probs.cpu())

y_true = torch.cat(all_true).numpy()
y_pred = torch.cat(all_pred).numpy()
y_probs = torch.cat(all_probs).numpy()
# Confianza = probabilidad asignada a la clase predicha
y_conf = y_probs.max(axis=1)

print(f'Total de clips en test: {len(y_true)}')
print(f'Accuracy global: {(y_true == y_pred).mean():.4f}  (esperado ~0.632)')
print(f'Shape de probabilidades: {y_probs.shape}  (clips, clases)')

---
## Sección 3 — Matriz de confusión normalizada por fila

### El problema de la matriz cruda

Ya tienes una matriz de confusión con conteos absolutos (`confusion_matrix.png`). El problema es que el desbalance la distorsiona: `no_ave` tiene 240 clips en test y `Rupornis_magnirostris` solo 14. Una celda con "50" significa cosas muy distintas según la clase. Es imposible comparar el comportamiento del modelo entre clases mirando conteos.

### La solución: normalizar por fila (recall por clase)

Si dividimos cada fila por su suma (el total de clips reales de esa clase), cada celda pasa a ser una **proporción**: *de todos los clips que realmente eran de la especie X, qué fracción se predijo como Y*. La diagonal pasa a ser exactamente el **recall** de cada clase, y las celdas fuera de la diagonal nos dicen **hacia dónde se fugan** los errores.

Esto pone a todas las clases en la misma escala [0, 1], y hace el patrón de confusión legible de un vistazo.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
# Normalizar por fila (evitar división por cero con np.errstate)
with np.errstate(all='ignore'):
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)

# Nombres cortos para que quepan en los ejes
short_names = [n[:14] for n in class_names]

fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor('#FAFAFA')
im = ax.imshow(cm_norm, cmap='Purples', vmin=0, vmax=1)

ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(short_names, fontsize=9)
ax.set_xlabel('Predicción del modelo', fontsize=12, color=COLOR_DARK)
ax.set_ylabel('Especie real', fontsize=12, color=COLOR_DARK)
ax.set_title('Matriz de confusión normalizada por fila (recall)\nDiagonal = recall por clase',
             fontsize=13, color=COLOR_DARK)

# Anotar valores
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        val = cm_norm[i, j]
        if val >= 0.01:
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    color='white' if val > 0.5 else COLOR_DARK, fontsize=8)

fig.colorbar(im, ax=ax, label='Proporción', fraction=0.046, pad=0.04)
plt.tight_layout()
out_path = RUN_DIR / 'confusion_matrix_normalized.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
print(f'Guardada: {out_path}')
plt.show()

In [ ]:
# Extraer automáticamente las confusiones más fuertes (fuera de la diagonal)
print('CONFUSIONES MÁS FUERTES (fuera de la diagonal):\n')
confusiones = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm_norm[i, j] > 0:
            confusiones.append((cm_norm[i, j], class_names[i], class_names[j], cm[i, j]))
confusiones.sort(reverse=True)
for prop, real, pred, count in confusiones[:12]:
    print(f'  {prop:5.1%} de "{real}" se predijo como "{pred}"  ({count} clips)')

### Cómo leer esta matriz

- **Diagonal fuerte (oscura)** = clase bien clasificada (alto recall). Espera ver `no_ave` y `Crypturellus_cinereus` brillando aquí.
- **Celdas oscuras fuera de la diagonal** = confusiones sistemáticas. Estas son las que vamos a interpretar biológicamente en la Sección 6.
- **Filas "borrosas" (sin un valor dominante)** = clases que el modelo no logra capturar; sus errores se reparten entre varias especies. Típicamente son las clases con pocos datos.

La lista de arriba extrae automáticamente las confusiones más fuertes para que no tengas que leerlas a ojo. Guárdalas mentalmente: las vamos a explicar en la Sección 6.

---
## Sección 4 — F1 vs cantidad de datos: la hipótesis del desbalance

### La hipótesis

Nuestra sospecha principal (del análisis preliminar) es que **el rendimiento por clase está dominado por cuántos datos tiene cada clase**. Las especies con 20 archivos rinden mal; las que tienen 48-79 rinden bien; `no_ave` con 1600 archivos rinde casi perfecto.

Si esto es cierto, lo veremos como una **correlación positiva** entre el número de archivos de entrenamiento de cada clase y su F1-score en test. Vamos a graficarlo y calcular el coeficiente de correlación de Pearson para cuantificarlo.

### Por qué importa para el proyecto

Si el desbalance es el cuello de botella, entonces la palanca más poderosa **no** es la arquitectura (attention) sino los **datos**: conseguir más grabaciones de las especies raras, o usar técnicas de balanceo (oversampling, class weights, augmentation dirigida). Esto reorienta la estrategia del proyecto basándonos en evidencia, no en intuición.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

# F1 por clase
f1_per_class = f1_score(y_true, y_pred, labels=range(NUM_CLASSES), average=None, zero_division=0)
prec_per_class = precision_score(y_true, y_pred, labels=range(NUM_CLASSES), average=None, zero_division=0)
rec_per_class = recall_score(y_true, y_pred, labels=range(NUM_CLASSES), average=None, zero_division=0)

# Soporte en test (cuántos clips reales hay de cada clase)
support_test = np.array([(y_true == i).sum() for i in range(NUM_CLASSES)])

# Número de ARCHIVOS por clase en train (del log de entrenamiento).
# Estos valores vienen del output de build_index durante el entrenamiento.
archivos_por_clase = {
    'no_ave': 1600,
    'Celeus_grammicus': 28,
    'Chordeiles_pusillus': 21,
    'Crypturellus_cinereus': 48,
    'Crypturellus_undulatus': 29,
    'Frederickena_fulva': 20,
    'Glaucidium_brasilianum': 22,
    'Lipaugus_vociferans': 36,
    'Ramphastos_tucanus': 31,
    'Rupornis_magnirostris': 20,
    'Trogon_viridis': 79,
}
n_archivos = np.array([archivos_por_clase[n] for n in class_names])

print('MÉTRICAS POR CLASE (ordenadas por F1):\n')
print(f'{"Clase":<24} {"Archivos":>8} {"Support":>8} {"Prec":>6} {"Recall":>7} {"F1":>6}')
print('-' * 64)
orden = np.argsort(f1_per_class)
for i in orden:
    print(f'{class_names[i]:<24} {n_archivos[i]:>8} {support_test[i]:>8} '
          f'{prec_per_class[i]:>6.2f} {rec_per_class[i]:>7.2f} {f1_per_class[i]:>6.2f}')

In [ ]:
# Correlación entre número de archivos y F1 (excluyendo no_ave que es un outlier extremo)
mask_aves = np.array([n != 'no_ave' for n in class_names])

corr_con_noave = np.corrcoef(n_archivos, f1_per_class)[0, 1]
corr_sin_noave = np.corrcoef(n_archivos[mask_aves], f1_per_class[mask_aves])[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor('#FAFAFA')

# Panel izquierdo: con no_ave (escala log porque 1600 vs 20)
axes[0].scatter(n_archivos, f1_per_class, s=120, c=COLOR_PRIMARY, alpha=0.7, edgecolors=COLOR_DARK)
for i in range(NUM_CLASSES):
    axes[0].annotate(class_names[i][:10], (n_archivos[i], f1_per_class[i]),
                     fontsize=7, alpha=0.8, xytext=(5, 5), textcoords='offset points')
axes[0].set_xscale('log')
axes[0].set_xlabel('Archivos de entrenamiento (escala log)', fontsize=11, color=COLOR_DARK)
axes[0].set_ylabel('F1-score en test', fontsize=11, color=COLOR_DARK)
axes[0].set_title(f'Todas las clases (Pearson r = {corr_con_noave:.3f})', fontsize=12, color=COLOR_DARK)
axes[0].grid(alpha=0.3)

# Panel derecho: solo aves (sin no_ave)
axes[1].scatter(n_archivos[mask_aves], f1_per_class[mask_aves], s=120, c=COLOR_ACCENT,
                alpha=0.7, edgecolors=COLOR_DARK)
for i in np.where(mask_aves)[0]:
    axes[1].annotate(class_names[i][:10], (n_archivos[i], f1_per_class[i]),
                     fontsize=7, alpha=0.8, xytext=(5, 5), textcoords='offset points')
# Línea de tendencia
z = np.polyfit(n_archivos[mask_aves], f1_per_class[mask_aves], 1)
xs = np.linspace(n_archivos[mask_aves].min(), n_archivos[mask_aves].max(), 50)
axes[1].plot(xs, np.poly1d(z)(xs), '--', color=COLOR_WARN, lw=2, label='Tendencia lineal')
axes[1].set_xlabel('Archivos de entrenamiento', fontsize=11, color=COLOR_DARK)
axes[1].set_ylabel('F1-score en test', fontsize=11, color=COLOR_DARK)
axes[1].set_title(f'Solo aves, sin no_ave (Pearson r = {corr_sin_noave:.3f})', fontsize=12, color=COLOR_DARK)
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
out_path = RUN_DIR / 'f1_vs_datos.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
print(f'Guardada: {out_path}')
plt.show()

print(f'\nCorrelación de Pearson (todas):    r = {corr_con_noave:.3f}')
print(f'Correlación de Pearson (solo aves): r = {corr_sin_noave:.3f}')

### Cómo interpretar la correlación

El coeficiente de Pearson *r* mide la fuerza de la relación lineal entre cantidad de datos y F1, en el rango [-1, 1]:

- *r* cercano a **+1**: relación positiva fuerte → más datos predicen claramente mejor F1. Confirma la hipótesis del desbalance.
- *r* cercano a **0**: no hay relación lineal → el desbalance no explica el rendimiento, habría que buscar otra causa (calidad de audio, solapamiento espectral entre especies, etc.).

Separamos el análisis en dos paneles a propósito: `no_ave` con 1600 archivos es un punto extremo que, por sí solo, puede inflar artificialmente la correlación. El panel derecho (solo aves) es el más honesto para juzgar si entre las especies de aves —que están en un rango comparable de 20 a 79 archivos— el patrón se mantiene.

> **Nota estadística:** con solo 10 especies, esta correlación es **indicativa, no concluyente**. Una sola especie atípica puede mover *r* bastante. Lo reportamos como evidencia que apoya la hipótesis, no como prueba definitiva.

---
## Sección 5 — Errores de alta confianza: los fallos más "vergonzosos"

### La idea

No todos los errores son iguales. Un error donde el modelo predijo la clase equivocada con probabilidad 0.35 (estaba dudando) es comprensible. Un error donde predijo la clase equivocada con probabilidad 0.95 (estaba *convencido*) es mucho más informativo y preocupante: significa que el modelo aprendió un patrón espurio que lo lleva a equivocarse con seguridad.

Estos errores de alta confianza son oro para el diagnóstico porque suelen revelar:
- Solapamiento espectral real entre dos especies (problema del dominio, no del modelo).
- Etiquetas ruidosas en los datos (un clip mal etiquetado en Xeno-canto).
- Segmentos de silencio o ruido dentro de una grabación de ave, que se parecen más a `no_ave`.

In [ ]:
# Identificar errores y ordenarlos por confianza
errores_mask = (y_true != y_pred)
idx_errores = np.where(errores_mask)[0]

# Ordenar por confianza descendente
idx_errores_ordenados = idx_errores[np.argsort(-y_conf[idx_errores])]

print(f'Total de errores: {errores_mask.sum()} de {len(y_true)} ({errores_mask.mean():.1%})\n')
print('TOP 15 ERRORES DE MAYOR CONFIANZA:\n')
print(f'{"Confianza":>9} {"Real":<24} {"Predicho":<24}')
print('-' * 60)
for idx in idx_errores_ordenados[:15]:
    real = class_names[y_true[idx]]
    pred = class_names[y_pred[idx]]
    print(f'{y_conf[idx]:>9.1%} {real:<24} {pred:<24}')

In [ ]:
# Distribución de confianza: aciertos vs errores
fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor('#FAFAFA')

conf_aciertos = y_conf[~errores_mask]
conf_errores = y_conf[errores_mask]

bins = np.linspace(0, 1, 30)
ax.hist(conf_aciertos, bins=bins, alpha=0.6, label=f'Aciertos (n={len(conf_aciertos)})',
        color=COLOR_ACCENT, edgecolor='white')
ax.hist(conf_errores, bins=bins, alpha=0.6, label=f'Errores (n={len(conf_errores)})',
        color=COLOR_WARN, edgecolor='white')
ax.axvline(1.0 / NUM_CLASSES, color=COLOR_DARK, ls=':', alpha=0.7,
           label=f'Azar (1/{NUM_CLASSES} = {1/NUM_CLASSES:.2f})')
ax.set_xlabel('Confianza del modelo (prob. de la clase predicha)', fontsize=11, color=COLOR_DARK)
ax.set_ylabel('Número de clips', fontsize=11, color=COLOR_DARK)
ax.set_title('Distribución de confianza: aciertos vs errores', fontsize=13, color=COLOR_DARK)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
out_path = RUN_DIR / 'distribucion_confianza.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
print(f'Guardada: {out_path}')
plt.show()

print(f'\nConfianza media en aciertos: {conf_aciertos.mean():.3f}')
print(f'Confianza media en errores:  {conf_errores.mean():.3f}')

### Qué buscar en este histograma

Lo ideal sería ver los **errores concentrados en confianza baja** (el modelo duda cuando se equivoca) y los **aciertos en confianza alta** (el modelo está seguro cuando acierta). En la práctica casi siempre hay solapamiento.

- Si los errores tienen confianza media **alta** (cercana a la de los aciertos), el modelo está **mal calibrado**: no sabe cuándo no sabe. Esto complica fijar el umbral de "No identificado".
- Si los errores tienen confianza claramente **más baja** que los aciertos, hay margen para que un umbral de confianza filtre buena parte de los errores convirtiéndolos en "No identificado". Esto lo cuantificamos en la Sección 7.

---
## Sección 6 — Interpretación bioacústica de las confusiones

### De la métrica al fenómeno

Aquí conectamos los números con la biología. Las confusiones más fuertes (Sección 3) se pueden clasificar en tres tipos:

**1. Confusiones taxonómicamente razonables ("errores honestos").**
Especies del mismo género suelen tener vocalizaciones emparentadas. En nuestro dataset, *Crypturellus cinereus* y *Crypturellus undulatus* son ambos tinamúes del género *Crypturellus*. Sus cantos son silbidos graves y modulados, fáciles de confundir incluso para humanos. Si el modelo los confunde entre sí, es un error esperable y "defendible".

**2. Fuga hacia `no_ave` (segmentos sin canto).**
Nuestras grabaciones de Xeno-canto duran a veces decenas de segundos, pero el ave no canta todo el tiempo. Al segmentar en clips de 5 s, algunos clips caen en tramos de silencio o ruido de fondo. Esos clips, aunque etiquetados como la especie, *acústicamente* son ruido → el modelo los manda a `no_ave`. No es un error del modelo, es una limitación de la **segmentación a ciegas** (sin detección de actividad vocal).

**3. Fuga hacia la clase mayoritaria (sesgo de prior).**
Cuando una clase tiene poquísimos datos, el modelo aprende un *prior* débil para ella y tiende a "refugiarse" en clases con más representación. Es el clásico efecto del desbalance.

La siguiente celda etiqueta automáticamente cada confusión fuerte según estas categorías.

In [ ]:
# Diccionario de géneros para detectar confusiones intra-género
def genero(nombre):
    """Extrae el género (primera palabra del nombre científico)."""
    if nombre == 'no_ave':
        return 'no_ave'
    return nombre.split('_')[0]

print('CLASIFICACIÓN DE LAS CONFUSIONES MÁS FUERTES:\n')
for prop, real, pred, count in confusiones[:12]:
    # Determinar el tipo de confusión
    if pred == 'no_ave':
        tipo = '[FUGA A no_ave]  -> probable segmento de silencio/ruido sin canto'
    elif genero(real) == genero(pred):
        tipo = '[MISMO GÉNERO]   -> confusión taxonómicamente razonable'
    else:
        tipo = '[INTER-GÉNERO]   -> revisar solapamiento espectral o sesgo de datos'
    print(f'  {prop:5.1%}  {real}  ->  {pred}')
    print(f'         {tipo}\n')

### Lectura del resultado

Esta clasificación automática te da el guion para el reporte. En lugar de decir "el modelo confunde A con B", podrás escribir frases con criterio de dominio como:

> "El X% de las confusiones más fuertes ocurren entre especies del mismo género, lo que indica que el modelo captura correctamente la estructura acústica general pero no discrimina diferencias finas intra-género — justamente el tipo de detalle que un mecanismo de atención podría aprender a resaltar."

Y respecto a las fugas hacia `no_ave`:

> "Una fracción de los errores corresponde a clips etiquetados como ave pero que acústicamente son silencio o ruido, consecuencia de segmentar sin detección de actividad vocal (VAD). Esto sugiere que un preprocesamiento con VAD podría mejorar el dataset antes de cambiar la arquitectura."

---
## Sección 7 — Calibración y el umbral de "No identificado"

### Conexión directa con el diseño del proyecto

SelvaSonic incluye por diseño una clase de rechazo: cuando la confianza del modelo cae por debajo de un umbral, en vez de arriesgar una predicción, el sistema responde **"No identificado"**. Esto es esencial en un sistema real de campo: es preferible que el sistema admita ignorancia a que afirme con seguridad una especie equivocada.

La pregunta de ingeniería es: **¿qué umbral elegir?** Un umbral muy alto rechaza demasiado (pierde aves reales); uno muy bajo deja pasar errores. Vamos a simular barriendo el umbral de 0 a 1 y midiendo, en cada punto, qué pasaría con los clips de test.

Este análisis es preparatorio para la Semana 5-6, pero lo adelantamos porque los datos ya están aquí.

In [ ]:
umbrales = np.linspace(0, 0.99, 100)
cobertura = []      # fracción de clips que SÍ se clasifican (confianza >= umbral)
acc_aceptados = []  # accuracy SOLO sobre los clips aceptados

for u in umbrales:
    aceptados = y_conf >= u
    cobertura.append(aceptados.mean())
    if aceptados.sum() > 0:
        acc_aceptados.append((y_true[aceptados] == y_pred[aceptados]).mean())
    else:
        acc_aceptados.append(np.nan)

cobertura = np.array(cobertura)
acc_aceptados = np.array(acc_aceptados)

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor('#FAFAFA')
ax.plot(umbrales, cobertura, color=COLOR_PRIMARY, lw=2.5, label='Cobertura (clips clasificados)')
ax.plot(umbrales, acc_aceptados, color=COLOR_ACCENT, lw=2.5, label='Accuracy en clips aceptados')
ax.set_xlabel('Umbral de confianza', fontsize=11, color=COLOR_DARK)
ax.set_ylabel('Proporción', fontsize=11, color=COLOR_DARK)
ax.set_title('Trade-off cobertura vs precisión al variar el umbral de "No identificado"',
             fontsize=12, color=COLOR_DARK)
ax.legend(); ax.grid(alpha=0.3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
plt.tight_layout()
out_path = RUN_DIR / 'umbral_no_identificado.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
print(f'Guardada: {out_path}')
plt.show()

# Reportar algunos umbrales de referencia
print('\nTRADE-OFF EN UMBRALES DE REFERENCIA:\n')
print(f'{"Umbral":>7} {"Cobertura":>10} {"Acc aceptados":>14}')
print('-' * 33)
for u_ref in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9]:
    idx_u = np.argmin(np.abs(umbrales - u_ref))
    print(f'{u_ref:>7.2f} {cobertura[idx_u]:>10.1%} {acc_aceptados[idx_u]:>14.1%}')

### Cómo usar esta gráfica

Las dos curvas van en direcciones opuestas, y ahí está el compromiso:

- Al **subir el umbral**, la **cobertura baja** (clasificamos menos clips, mandamos más a "No identificado") pero la **accuracy de los que sí clasificamos sube** (porque solo nos quedamos con las predicciones más seguras).
- El punto óptimo depende de la prioridad del proyecto: ¿preferimos identificar muchas aves aunque con más errores, o identificar pocas pero con alta certeza?

Para un sistema de monitoreo de biodiversidad, normalmente se prioriza la **precisión** (no "inventar" especies), así que un umbral relativamente alto (0.7-0.8) suele tener sentido. La tabla te da los números exactos para justificar la elección en el reporte. El valor `CONFIDENCE_THRESHOLD = 0.6` que dejaste reservado en `config.py` es un punto de partida razonable que ahora puedes ajustar con evidencia.

---
## Sección 8 — Conclusiones y hoja de ruta para Semana 4

La siguiente celda genera un resumen cuantitativo automático que consolida todo el análisis y lo guarda en un archivo de texto para el reporte.

In [ ]:
from sklearn.metrics import f1_score as _f1

macro_f1 = _f1(y_true, y_pred, average='macro', zero_division=0)
weighted_f1 = _f1(y_true, y_pred, average='weighted', zero_division=0)
acc_global = (y_true == y_pred).mean()

# Clases por encima/debajo de F1 = 0.5
buenas = [class_names[i] for i in range(NUM_CLASSES) if f1_per_class[i] >= 0.5]
malas = [class_names[i] for i in range(NUM_CLASSES) if f1_per_class[i] < 0.5]

# Fracción de errores que van a no_ave
noave_idx = label_map['no_ave']
errores_a_noave = ((y_true != y_pred) & (y_pred == noave_idx)).sum()
total_errores = (y_true != y_pred).sum()

resumen = f"""
{'=' * 70}
RESUMEN DEL ANÁLISIS DE ERRORES — BASELINE SelvaSonic S3
{'=' * 70}

MÉTRICAS GLOBALES
  Accuracy global         : {acc_global:.4f}
  Macro F1 (no ponderado) : {macro_f1:.4f}
  Weighted F1             : {weighted_f1:.4f}
  Baseline aleatorio      : {1/NUM_CLASSES:.4f}  (1/{NUM_CLASSES})
  Mejora sobre azar       : {acc_global / (1/NUM_CLASSES):.1f}x

RENDIMIENTO POR CLASE
  Clases con F1 >= 0.5 ({len(buenas)}): {', '.join(buenas)}
  Clases con F1 <  0.5 ({len(malas)}): {', '.join(malas)}

PATRÓN DE ERRORES
  Total de errores        : {total_errores} de {len(y_true)} ({total_errores/len(y_true):.1%})
  Errores que van a no_ave : {errores_a_noave} ({errores_a_noave/total_errores:.1%} de los errores)
  Confianza media aciertos : {conf_aciertos.mean():.3f}
  Confianza media errores  : {conf_errores.mean():.3f}

CORRELACIÓN DATOS-RENDIMIENTO
  Pearson r (todas)        : {corr_con_noave:.3f}
  Pearson r (solo aves)    : {corr_sin_noave:.3f}

DIAGNÓSTICO
  - El modelo supera ampliamente el azar: la arquitectura y el pipeline funcionan.
  - El rendimiento es muy desigual entre clases, dominado por el desbalance de datos.
  - Las clases con mas datos (no_ave, Crypturellus_cinereus, Trogon_viridis,
    Lipaugus_vociferans) rinden bien; las de ~20 archivos rinden mal.
  - Hay un gap train/val (~0.97 vs ~0.74) que confirma overfitting moderado.

HOJA DE RUTA PARA SEMANA 4 (en orden de impacto esperado)
  1. DATOS (mayor palanca): balanceo de clases — class weights en la loss,
     u oversampling de especies raras. El desbalance es el cuello de botella #1.
  2. SEGMENTACION: explorar deteccion de actividad vocal (VAD) para no generar
     clips de silencio etiquetados como ave (reduce la fuga hacia no_ave).
  3. ARQUITECTURA: agregar Multi-Head Self-Attention. Hipotesis a verificar:
     el attention ayudara sobre todo a discriminar especies del mismo genero,
     resaltando las regiones tiempo-frecuencia mas informativas del espectrograma.
  4. REGULARIZACION: si el overfitting persiste, subir dropout o data augmentation.

  Metrica clave a vigilar en S4: el MACRO F1 (no el accuracy), porque es el que
  refleja si mejoramos en las clases dificiles y no solo en no_ave.
{'=' * 70}
"""

print(resumen)

# Guardar a archivo para el reporte
out_path = RUN_DIR / 'analisis_errores_resumen.txt'
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(resumen)
print(f'\nResumen guardado en: {out_path}')

---
## Cierre

Este notebook respondió las dos preguntas del cronograma con evidencia cuantitativa:

**¿Qué confunde el modelo?**
- Confunde especies del mismo género entre sí (error razonable).
- Manda a `no_ave` clips de aves que en realidad son tramos de silencio/ruido.
- Falla sistemáticamente en las especies con pocos datos, fugándose hacia clases mayoritarias.

**¿Por qué?**
- El desbalance de datos es la causa dominante (correlación datos-F1).
- La segmentación a ciegas introduce clips sin canto.
- El overfitting moderado limita la generalización en clases pequeñas.

Los artefactos generados (matrices, gráficas y el resumen `.txt`) quedan en la carpeta del run, listos para el reporte y para comparar contra el modelo con attention de la Semana 4.

**Siguiente paso:** commit de este notebook y de los artefactos generados, y arranque de la Semana 4.